# Chapter 6 — Math and Logic Puzzles

Most of these are reasoning questions, not coding ones. For each, we give the reasoning in brief and then **verify it with code**: a simulation, a brute-force search, or a direct computation.

Background: primes and the Sieve of Eratosthenes, and basic probability (`P(A and B) = P(B|A)P(A)`, `P(A or B) = P(A) + P(B) - P(A and B)`).

In [1]:
import itertools
import math
import random
from collections import Counter, deque
from fractions import Fraction


def is_prime(n: int) -> bool:
    if n < 2:
        return False
    return all(n % i for i in range(2, math.isqrt(n) + 1))


def sieve(limit: int) -> list[int]:
    flags = bytearray([1]) * (limit + 1)
    flags[0:2] = b"\x00\x00"
    for p in range(2, math.isqrt(limit) + 1):
        if flags[p]:
            flags[p * p::p] = bytearray(len(range(p * p, limit + 1, p)))
    return [i for i, f in enumerate(flags) if f]


assert sieve(50) == [p for p in range(51) if is_prime(p)]
print(sieve(50))

[2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47]


## 6.1 The Heavy Pill
Twenty bottles of pills: one bottle has 1.1 g pills, the rest 1.0 g. A scale may be used **once**. Take `i` pills from bottle `i` (1..20), weigh them all: the heavy bottle is `(weight - 210) / 0.1`.

In [2]:
def find_heavy_bottle(bottle_weights: list[float]) -> int:
    total = sum(i * w for i, w in enumerate(bottle_weights, start=1))  # the single weighing
    expected = sum(range(1, len(bottle_weights) + 1))
    return round((total - expected) / 0.1)


for heavy in range(1, 21):
    weights = [1.0] * 20
    weights[heavy - 1] = 1.1
    assert find_heavy_bottle(weights) == heavy
print("6.1 ok")

6.1 ok


## 6.2 Basketball
Game 1: one shot. Game 2: make at least 2 of 3. For which make-probability `p` should you pick each?
`P(game2) = 3p²(1-p) + p³ = 3p² - 2p³`. Game 1 is better when `p > 3p² - 2p³` ⇔ `(2p - 1)(p - 1) > 0` ⇔ `p < 0.5`. (Equal at 0, 0.5, 1.)

In [3]:
def p_game2(p: float) -> float:
    return 3 * p**2 * (1 - p) + p**3


def simulate_game2(p: float, trials: int = 100_000) -> float:
    return sum(sum(random.random() < p for _ in range(3)) >= 2 for _ in range(trials)) / trials


for p in [0.2, 0.5, 0.8]:
    print(f"p={p}: game1={p:.3f} game2={p_game2(p):.3f} (sim {simulate_game2(p):.3f}) -> "
          f"{'game 1' if p > p_game2(p) else 'game 2' if p < p_game2(p) else 'either'}")

p=0.2: game1=0.200 game2=0.104 (sim 0.104) -> game 1
p=0.5: game1=0.500 game2=0.500 (sim 0.500) -> either
p=0.8: game1=0.800 game2=0.896 (sim 0.897) -> game 2


## 6.3 Dominoes
An 8×8 board with two diagonally opposite corners removed; can 31 dominoes cover it? **No**: both removed corners are the same colour, leaving 30 of one colour and 32 of the other, but every domino covers one of each. We verify with a brute-force tiling search on smaller boards.

In [4]:
def can_tile(n: int, removed: set[tuple[int, int]]) -> bool:
    cells = frozenset((r, c) for r in range(n) for c in range(n)) - removed

    from functools import cache

    @cache
    def solve(remaining: frozenset) -> bool:
        if not remaining:
            return True
        r, c = min(remaining)  # first uncovered cell must be covered right or down
        for dr, dc in ((0, 1), (1, 0)):
            other = (r + dr, c + dc)
            if other in remaining:
                if solve(remaining - {(r, c), other}):
                    return True
        return False

    return solve(cells)


for n in (2, 4, 6):
    assert can_tile(n, set())
    assert not can_tile(n, {(0, 0), (n - 1, n - 1)})
print("6.3 ok: opposite-corner boards are never tileable")

6.3 ok: opposite-corner boards are never tileable


## 6.4 Ants on a Triangle
Three ants on the corners of a triangle each pick a random edge direction. Probability of a collision? No collision only if all go clockwise or all counterclockwise: `2 / 2³ = 1/4`, so collision = `3/4`. For an n-gon: `1 - 2 / 2ⁿ = 1 - (1/2)^(n-1)`.

In [5]:
def collision_probability(n: int) -> Fraction:
    no_collision = sum(1 for dirs in itertools.product((0, 1), repeat=n) if len(set(dirs)) == 1)
    return 1 - Fraction(no_collision, 2**n)


assert collision_probability(3) == Fraction(3, 4)
assert all(collision_probability(n) == 1 - Fraction(1, 2 ** (n - 1)) for n in range(3, 10))
print("6.4 ok")

6.4 ok


## 6.5 Jugs of Water
With a 5-quart and a 3-quart jug (unlimited water), measure exactly 4 quarts. BFS over `(a, b)` states finds the shortest sequence.

In [6]:
def jug_solution(cap_a: int, cap_b: int, target: int) -> list[tuple[int, int]] | None:
    start = (0, 0)
    parent: dict[tuple[int, int], tuple[int, int] | None] = {start: None}
    q = deque([start])
    while q:
        a, b = q.popleft()
        if a == target or b == target:
            path = []
            state: tuple[int, int] | None = (a, b)
            while state is not None:
                path.append(state)
                state = parent[state]
            return path[::-1]
        pour_ab = min(a, cap_b - b)
        pour_ba = min(b, cap_a - a)
        for nxt in [(cap_a, b), (a, cap_b), (0, b), (a, 0),
                    (a - pour_ab, b + pour_ab), (a + pour_ba, b - pour_ba)]:
            if nxt not in parent:
                parent[nxt] = (a, b)
                q.append(nxt)
    return None


path = jug_solution(5, 3, 4)
print(" -> ".join(map(str, path)))
assert path[-1][0] == 4

(0, 0) -> (5, 0) -> (2, 3) -> (2, 0) -> (0, 2) -> (5, 2) -> (4, 3)


## 6.6 Blue-Eyed Island
Everyone on an island is told at least one person has blue eyes. Blue-eyed people must leave on a nightly flight as soon as they *know*. Nobody can see their own eyes, and nobody talks. How long until they leave? **With `c` blue-eyed people, all of them leave on night `c`.** Induction: with c=1, that person sees no blue eyes and leaves night 1. With c=2, each sees one blue-eyed person, who doesn't leave on night 1, so each concludes they are blue-eyed and both leave on night 2. The same argument continues upward.

We simulate this with a knowledge model: a blue-eyed person who sees `k` blue-eyed people leaves on night `k+1`, unless those `k` have already left by night `k`.

In [7]:
def blue_eyed_departure(num_people: int, blue: set[int]) -> int:
    for night in itertools.count(1):
        # person i sees count of blue others; they can deduce on night (seen + 1)
        leaving = {i for i in blue if len(blue - {i}) + 1 == night}
        if leaving:
            return night


for c in range(1, 8):
    assert blue_eyed_departure(100, set(range(c))) == c
print("6.6 ok: c blue-eyed people leave on night c")

6.6 ok: c blue-eyed people leave on night c


## 6.7 The Apocalypse
A queen decrees every family must keep having children until they have one girl. What's the gender ratio? **Still 50/50.** Every birth is an independent fair coin flip, and a stopping rule can't change the expected ratio. Simulation below.

In [8]:
def simulate_families(n: int) -> float:
    girls = boys = 0
    for _ in range(n):
        while random.random() < 0.5:
            boys += 1
        girls += 1
    return girls / (girls + boys)


ratio = simulate_families(200_000)
print(f"girls fraction ≈ {ratio:.4f}")
assert abs(ratio - 0.5) < 0.01

girls fraction ≈ 0.4992


## 6.8 The Egg Drop Problem
A 100-floor building and 2 eggs: minimise the worst-case number of drops. Drop egg 1 at intervals that shrink by 1 each time (14, 27, 39, …), so that total worst case stays constant: `x + (x-1) + … + 1 ≥ 100` → `x = 14`. General DP verifies it.

In [9]:
from functools import cache


@cache
def egg_drop(eggs: int, floors: int) -> int:
    """Worst-case drops needed to find the critical floor."""
    if floors == 0 or eggs == 1:
        return floors
    return 1 + min(max(egg_drop(eggs - 1, f - 1), egg_drop(eggs, floors - f)) for f in range(1, floors + 1))


assert egg_drop(2, 100) == 14
print("2 eggs, 100 floors:", egg_drop(2, 100), "| 3 eggs:", egg_drop(3, 100))


def simulate_14_strategy(break_floor: int) -> int:
    """Returns drops used by the decreasing-interval strategy; break_floor is the lowest floor that breaks (101 = never)."""
    drops, floor, step, last_safe = 0, 0, 14, 0
    while True:
        floor = min(floor + step, 100)
        drops += 1
        if floor >= break_floor:
            break
        last_safe = floor
        if floor == 100:
            return drops
        step -= 1
    for f in range(last_safe + 1, floor):  # linear scan with egg 2
        drops += 1
        if f >= break_floor:
            break
    return drops


assert max(simulate_14_strategy(b) for b in range(1, 102)) == 14
print("6.8 ok")

2 eggs, 100 floors: 14 | 3 eggs: 9
6.8 ok


## 6.9 100 Lockers
100 closed lockers; on pass `i`, toggle every i-th locker. After pass 100, how many are open? Locker `n` is toggled once per factor; only **perfect squares** have an odd number of factors → **10** open.

In [10]:
lockers = [False] * 101
for i in range(1, 101):
    for j in range(i, 101, i):
        lockers[j] = not lockers[j]
open_lockers = [n for n in range(1, 101) if lockers[n]]
assert open_lockers == [k * k for k in range(1, 11)]
print(open_lockers)

[1, 4, 9, 16, 25, 36, 49, 64, 81, 100]


## 6.10 Poison
1000 bottles of soda, one poisoned. Ten test strips each give a positive/negative result, but only after 7 days, and a positive strip can't be reused. Find the bottle in as few days as possible. **Binary encoding**: put a drop of bottle `b` on strip `i` whenever bit `i` of `b` is set, and run every test on day 0. The strips that turn positive spell out the bottle number in binary after 7 days (2¹⁰ = 1024 ≥ 1000).

In [11]:
def find_poison(poisoned: int, num_bottles: int = 1000, strips: int = 10) -> int:
    strip_drops: list[set[int]] = [set() for _ in range(strips)]
    for bottle in range(num_bottles):
        for i in range(strips):
            if bottle >> i & 1:
                strip_drops[i].add(bottle)
    positives = [poisoned in drops for drops in strip_drops]  # read on day 7
    return sum(1 << i for i, pos in enumerate(positives) if pos)


assert all(find_poison(b) == b for b in range(1000))
print("6.10 ok: 7 days, one round of tests")

6.10 ok: 7 days, one round of tests
